### Fake Math Equalities

Some mathematical expressions look equal because their numerical values are extremely close. This notebook searches through combinations of familiar constants and functions to find convincing near-equalities, then prints and compares the results.

In [66]:
import math
from math import e, pi, sqrt, log as ln, sin, cos, exp
from collections import defaultdict
from random import choice, randint
from multiprocessing import Process, Pipe

phi = (1 + 5 ** 0.5) / 2 # golden ratio

def calc(expr, precision=4):
    result = eval(expr, globals())
    print(f'{expr} = {result:0.{precision}f}')

calc("e ** pi - pi")
calc("e**6 - (pi**4 + pi**5)")
calc("sqrt(2) * ln(pi) - phi")
calc("ln(2) + ln(3) - ln(6)")
calc("ln(pi**2) - 2 * ln(pi)")
calc('cos((2 ** 10)) - cos(sqrt((10 ** phi)))')
calc("sqrt(3) * (6 + ln(pi)) - sqrt((150 + pi))")
calc("sin(phi) ** e - sin(sqrt(e))")
calc("cos(e * cos(1)) - sin(exp((e - 5)))")
print('------------------------------------------------------------')
calc("sin(9 * ln(4)) - sin(ln((8 ** 6) - ln(phi)))")
calc("((1 + sin(6)) ** 3) ** e - (ln(phi) - sin(9))")
calc("cos(2 - sqrt(10) + exp(e)) - sin(phi ** 8)")
calc("sin((9 ** pi) - sin(sqrt(4))) - sin((exp(2) * 1) - 6)")
calc("cos(13 * exp(10)) - cos(8 - ((3 + 10) * 5))")
calc("ln(sin(4) + 8) - sqrt(phi + ln(10))")
calc("(cos(7 + sin(10)) * sin(8)) - sin(sqrt(ln(5) + phi))")
calc("cos(sin(cos(5)) ** e) - cos(exp(cos(1) - 4))")
print('------------------------------------------------------------')
calc('cos(cos(phi)) - ln(sin(phi)) - 1')
calc('exp(cos(1)) - sin(exp(e)) - sqrt((cos(sqrt(pi)) + phi))')

# e^{\cos(1)} - \sin\left(e^e\right) \approx \sqrt{\cos\left(\sqrt{\pi}\right) + \phi}


e ** pi - pi = 19.9991
e**6 - (pi**4 + pi**5) = 0.0000
sqrt(2) * ln(pi) - phi = 0.0009
ln(2) + ln(3) - ln(6) = 0.0000
ln(pi**2) - 2 * ln(pi) = 0.0000
cos((2 ** 10)) - cos(sqrt((10 ** phi))) = -0.0001
sqrt(3) * (6 + ln(pi)) - sqrt((150 + pi)) = -0.0000
sin(phi) ** e - sin(sqrt(e)) = 0.0000
cos(e * cos(1)) - sin(exp((e - 5))) = -0.0000
------------------------------------------------------------
sin(9 * ln(4)) - sin(ln((8 ** 6) - ln(phi))) = 0.0000
((1 + sin(6)) ** 3) ** e - (ln(phi) - sin(9)) = 0.0000
cos(2 - sqrt(10) + exp(e)) - sin(phi ** 8) = 0.0000
sin((9 ** pi) - sin(sqrt(4))) - sin((exp(2) * 1) - 6) = 0.0000
cos(13 * exp(10)) - cos(8 - ((3 + 10) * 5)) = 0.0000
ln(sin(4) + 8) - sqrt(phi + ln(10)) = 0.0000
(cos(7 + sin(10)) * sin(8)) - sin(sqrt(ln(5) + phi)) = -0.0000
cos(sin(cos(5)) ** e) - cos(exp(cos(1) - 4)) = 0.0000
------------------------------------------------------------
cos(cos(phi)) - ln(sin(phi)) - 1 = 0.0000
exp(cos(1)) - sin(exp(e)) - sqrt((cos(sqrt(pi)) + phi)) = -0.

In [52]:
# Define the primitives and operations
primitives = ['1', '2', '3', 'e', 'pi', 'phi']
unary_ops = ['sqrt', 'ln', 'sin', 'cos', 'exp']
binary_ops = ['+', '-', '*', '**']

# Generate a random expression
def generate_expression(depth=0):
    if depth > 3 or (depth > 1 and choice([True, False])):
        return choice(primitives)
    if choice([True, False]):  # Unary operation
        return f"{choice(unary_ops)}({generate_expression(depth+1)})"
    else:  # Binary operation
        op = choice(binary_ops)
        if op == '**':
            right_depth = 999 # force a primitive
        else:
            right_depth = depth+1
        return f"({generate_expression(depth+1)} {op} {generate_expression(right_depth)})"

In [53]:
[ generate_expression() for _ in range(1_000) ];

In [55]:
# Multimap to store rounded results to expressions
result_to_expr = defaultdict(set)

In [56]:
# Generate and evaluate expressions
for _ in range(10000):
    expr = generate_expression()
    try:
        result = eval(expr, globals())
        if result is not None and not isinstance(result, complex) and not math.isnan(result) and not math.isinf(result):
            rounded_result = round(result, 5)
            result_to_expr[rounded_result].add(expr)
    except:
        pass

# Find keys in the multimap with more than one expression
candidates = {k: v for k, v in result_to_expr.items() if len(v) == 2}

# Second-pass filter to exclude exact matches
filtered_candidates = {}
for k, expressions in candidates.items():
    expr1, expr2 = list(expressions)
    result1 = eval(expr1, globals())
    result2 = eval(expr2, globals())
    
    if not math.isclose(result1, result2, rel_tol=1e-6):
        filtered_candidates[k] = expressions

# Display some of the candidates
list(filtered_candidates.items())  # Display only the first 10 to keep it concise

[(-0.00112, {'ln(cos(cos(phi)))', 'ln(sin(phi))'}),
 (1e-05, {'exp(((2 ** 3) - exp(3)))', 'exp(((pi - (3 * pi)) * sqrt(pi)))'}),
 (1.19069, {'(exp(cos(1)) - sin(exp(e)))', 'sqrt((cos(sqrt(pi)) + phi))'}),
 (0.85587, {'((cos(2) ** 1) + sqrt(phi))', 'sin(exp(((1 + e) + (phi - 1))))'}),
 (0.81671, {'((pi + e) - exp(phi))', 'cos(sin(cos((2 * e))))'}),
 (4e-05,
  {'((cos(sqrt(2)) ** 2) ** e)', 'exp(((2 * 3) - ((1 ** 3) + exp(e))))'}),
 (0.10544, {'cos((e * exp((1 - phi))))', 'cos((sqrt(pi) * e))'}),
 (-0.99871,
  {'cos(((2 - 2) + (3 + (pi ** phi))))', 'cos((2 - ((pi + 1) ** pi)))'}),
 (0.86126, {'(sqrt(sin(2)) ** pi)', 'cos((cos(e) + ((3 * 3) - sqrt(phi))))'}),
 (0.99999,
  {'cos((((3 - e) ** phi) ** e))', 'cos(((sin(phi) * 3) - (1 * 3)))'}),
 (0.99555, {'cos((1 * sin((phi * 2))))', 'cos((2 * sin(cos(phi))))'}),
 (-0.99981, {'cos(((pi ** 3) * pi))', 'sin((ln(2) + ((1 + 3) * (pi + 1))))'}),
 (1.71511,
  {'(cos((cos(1) ** 1)) * (2 ** 1))', 'sqrt((1 * ((phi + pi) * (phi - 1))))'}),
 (0.99109, 

In [21]:
calc('cos((2 ** 10)) - cos(sqrt((10 ** phi)))', precision=4)

cos((2 ** 10)) - cos(sqrt((10 ** phi))) = -0.0001
